<a href="https://colab.research.google.com/github/livinhaF/Distopico/blob/main/ProjetoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Configuração inicial**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Com certeza! Para iniciarmos nosso projeto, vamos criar a classe `DBManager` que será o ponto central de acesso ao nosso banco de dados SQLite. Ela garantirá a criação do arquivo `financas.db` e a estrutura inicial das tabelas `categorias`, `contas` e `transacoes`.

In [1]:
import sqlite3

class DBManager:
    def __init__(self, db_name='financas.db'):
        """
        Conecta ao banco de dados SQLite e cria as tabelas se não existirem.
        """
        self.db_name = db_name
        self.conn = None
        self.cursor = None
        self._connect()
        self._criar_tabelas()

    def _connect(self):
        """
        Estabelece a conexão com o banco de dados.
        """
        try:
            self.conn = sqlite3.connect(self.db_name)
            self.cursor = self.conn.cursor()
            print(f"Conectado ao banco de dados: {self.db_name}")
        except sqlite3.Error as e:
            print(f"Erro ao conectar ao banco de dados: {e}")

    def _criar_tabelas(self):
        """
        Cria as tabelas categorias, contas e transacoes.
        """
        if not self.conn:
            print("Erro: Conexão com o banco de dados não estabelecida.")
            return

        try:
            # Tabela categorias
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS categorias (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    nome TEXT UNIQUE NOT NULL,
                    tipo TEXT NOT NULL CHECK(tipo IN ('RECEITA', 'DESPESA'))
                );
            """)

            # Tabela contas
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS contas (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    nome TEXT UNIQUE NOT NULL,
                    saldo_inicial REAL NOT NULL
                );
            """)

            # Tabela transacoes
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS transacoes (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    id_conta INTEGER NOT NULL,
                    id_categoria INTEGER NOT NULL,
                    tipo TEXT NOT NULL CHECK(tipo IN ('RECEITA', 'DESPESA')),
                    valor REAL NOT NULL,
                    data TEXT NOT NULL,
                    descricao TEXT,
                    FOREIGN KEY (id_conta) REFERENCES contas(id) ON DELETE CASCADE,
                    FOREIGN KEY (id_categoria) REFERENCES categorias(id) ON DELETE CASCADE
                );
            """)

            self.conn.commit()
            print("Tabelas criadas ou já existentes.")
        except sqlite3.Error as e:
            print(f"Erro ao criar tabelas: {e}")

    def executar_consulta(self, sql, parametros=()):
        """
        Executa uma consulta SQL parametrizada e realiza o commit.
        Retorna o cursor para consultas SELECT.
        """
        if not self.conn:
            print("Erro: Conexão com o banco de dados não estabelecida.")
            return None

        try:
            self.cursor.execute(sql, parametros)
            self.conn.commit()
            return self.cursor
        except sqlite3.IntegrityError as e:
            print(f"Erro de integridade ao executar consulta: {e}")
            self.conn.rollback()
            return None
        except sqlite3.Error as e:
            print(f"Erro ao executar consulta: {e}")
            self.conn.rollback()
            return None

    def popular_dados_iniciais(self):
        """
        Popula o banco de dados com dados iniciais de exemplo.
        """
        print("Popolando dados iniciais...")
        # Inserir categorias
        categorias_iniciais = [
            ('Salário', 'RECEITA'),
            ('Aluguel', 'DESPESA'),
            ('Alimentação', 'DESPESA'),
            ('Transporte', 'DESPESA'),
            ('Investimentos', 'RECEITA')
        ]
        for nome, tipo in categorias_iniciais:
            sql = "INSERT OR IGNORE INTO categorias (nome, tipo) VALUES (?, ?)"
            self.executar_consulta(sql, (nome, tipo))

        # Inserir contas
        contas_iniciais = [
            ('Conta Corrente', 1500.00),
            ('Poupança', 5000.00)
        ]
        for nome, saldo in contas_iniciais:
            sql = "INSERT OR IGNORE INTO contas (nome, saldo_inicial) VALUES (?, ?)"
            self.executar_consulta(sql, (nome, saldo))
        print("Dados iniciais populados (se não existiam).")

    def inserir_transacao(self, id_conta, id_categoria, tipo, valor, data, descricao=None):
        """
        Insere uma nova transação no banco de dados.
        tipo: 'RECEITA' ou 'DESPESA'
        data: no formato 'YYYY-MM-DD'
        """
        sql = "INSERT INTO transacoes (id_conta, id_categoria, tipo, valor, data, descricao) VALUES (?, ?, ?, ?, ?, ?)"
        parametros = (id_conta, id_categoria, tipo, valor, data, descricao)
        cursor = self.executar_consulta(sql, parametros)
        if cursor:
            print(f"Transação registrada com sucesso! ID: {cursor.lastrowid}")
            return cursor.lastrowid
        return None

    def is_connected(self):
        """
        Verifica se a conexão com o banco de dados está ativa.
        """
        return self.conn is not None

    def close(self):
        """
        Fecha a conexão com o banco de dados.
        """
        if self.conn:
            self.conn.close()
            print(f"Conexão com {self.db_name} fechada.")

# Exemplo de uso:
# db_manager = DBManager('financas.db')
# db_manager.close()


Você pode testar a criação do banco de dados e das tabelas executando o código abaixo. Ele criará o arquivo `financas.db` no diretório atual (se ainda não existir) e imprimirá mensagens de status.

Depois de executar, você pode verificar a existência do arquivo `financas.db` e até mesmo inspecioná-lo com uma ferramenta como o DB Browser for SQLite.

In [2]:
# Para testar a classe:
db_manager = DBManager('financas.db')
db_manager.close()

Conectado ao banco de dados: financas.db
Tabelas criadas ou já existentes.
Conexão com financas.db fechada.


In [3]:
db_manager = DBManager('financas.db')
db_manager.close()

Conectado ao banco de dados: financas.db
Tabelas criadas ou já existentes.
Conexão com financas.db fechada.


Implementação dos métodos para inserção de dados e transação garantindo a parametrização

definição das classes

In [ ]:
class Categoria:
    def __init__(self, id, nome, tipo):
        self.id = id
        self.nome = nome
        self.tipo = tipo

    def __repr__(self):
        return f"Categoria(id={self.id}, nome='{self.nome}', tipo='{self.tipo}')"

class Transacao:
    def __init__(self, id, id_conta, id_categoria, tipo, valor, data, descricao=None):
        self.id = id
        self.id_conta = id_conta
        self.id_categoria = id_categoria
        self.tipo = tipo
        self.valor = valor
        self.data = data
        self.descricao = descricao

    def __repr__(self):
        return f"Transacao(id={self.id}, id_conta={self.id_conta}, id_categoria={self.id_categoria}, tipo='{self.tipo}', valor={self.valor}, data='{self.data}', descricao='{self.descricao}')"

Iterface gráfica parte 1 do trabalho

In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import sqlite3
from datetime import datetime

class AppGUI:
    def __init__(self, master):
        self.db = DBManager() # Instância do DBManager (POO - Responsabilidade)
        self.master = master
        master.title("Finança Simples V1 - Trabalho Final POO")
        master.geometry("850x650")

        # Variáveis de mapeamento de dados
        self.contas_map = {} # {nome: id}
        self.categorias_map = {} # {nome: id}

        self._carregar_dados_base() # Carrega os mapas para Comboboxes

        # Criação do Notebook (Abas)
        self.notebook = ttk.Notebook(master)
        self.notebook.pack(expand=True, fill='both', padx=10, pady=10)

        self._criar_aba_registro()
        self._criar_aba_categorias()
        self._criar_aba_relatorios()

    def _carregar_dados_base(self):
        """Popula os mapas de ID para nome, essenciais para as chaves estrangeiras."""
        # Contas
        self.contas_map = {nome: id for id, nome, saldo in self.db.buscar_contas()}
        # Categorias
        for id, nome, tipo in self.db.buscar_categorias():
            self.categorias_map[nome] = {'id': id, 'tipo': tipo}

    # --- ABA 1: REGISTRO DE TRANSAÇÕES ---

    def _criar_aba_registro(self):
        frame_registro = ttk.Frame(self.notebook, padding="15")
        self.notebook.add(frame_registro, text='💸 Registro de Transações')

        # Variáveis de controle
        self.tipo_transacao = tk.StringVar(value='RECEITA')
        self.valor_entry = tk.DoubleVar()
        self.data_entry = tk.StringVar(value=datetime.now().strftime('%Y-%m-%d'))
        self.descricao_entry = tk.StringVar()
        self.conta_selecionada = tk.StringVar()
        self.categoria_selecionada = tk.StringVar()

        # Função interna para gerar o layout mais fácil (Grid)
        def criar_campo(row, label_text, widget):
            ttk.Label(frame_registro, text=label_text + ":", font=('Arial', 10)).grid(row=row, column=0, sticky='w', pady=5, padx=5)
            widget.grid(row=row, column=1, sticky='ew', pady=5, padx=5, columnspan=2)

        # 0. Tipo (Radio Buttons)
        ttk.Radiobutton(frame_registro, text='Receita', variable=self.tipo_transacao, value='RECEITA').grid(row=0, column=0, sticky='w', pady=5)
        ttk.Radiobutton(frame_registro, text='Despesa', variable=self.tipo_transacao, value='DESPESA').grid(row=0, column=1, sticky='w', pady=5)

        # 1. Valor
        valor_widget = ttk.Entry(frame_registro, textvariable=self.valor_entry)
        criar_campo(1, "Valor (R$)", valor_widget)

        # 2. Data
        data_widget = ttk.Entry(frame_registro, textvariable=self.data_entry)
        criar_campo(2, "Data (YYYY-MM-DD)", data_widget)

        # 3. Conta (Combobox dinâmico)
        contas_nomes = list(self.contas_map.keys())
        conta_combo = ttk.Combobox(frame_registro, textvariable=self.conta_selecionada, values=contas_nomes, state='readonly')
        criar_campo(3, "Conta", conta_combo)
        if contas_nomes:
             conta_combo.set(contas_nomes[0])

        # 4. Categoria (Combobox dinâmico)
        categorias_nomes = list(self.categorias_map.keys())
        categoria_combo = ttk.Combobox(frame_registro, textvariable=self.categoria_selecionada, values=categorias_nomes, state='readonly')
        criar_campo(4, "Categoria", categoria_combo)
        if categorias_nomes:
            categoria_combo.set(categorias_nomes[0])

        # 5. Descrição
        descricao_widget = ttk.Entry(frame_registro, textvariable=self.descricao_entry)
        criar_campo(5, "Descrição", descricao_widget)

        # Botão
        ttk.Button(frame_registro, text="Adicionar Transação", command=self._adicionar_transacao).grid(row=6, column=0, columnspan=3, pady=15)

        frame_registro.columnconfigure(1, weight=1) # Faz a coluna de inputs expandir

    def _adicionar_transacao(self):
        """Valida e insere a transação no DB."""
        try:
            valor = self.valor_entry.get()
            data = self.data_entry.get()
            tipo = self.tipo_transacao.get()
            descricao = self.descricao_entry.get()

            # Mapeamento para IDs (Chaves Estrangeiras)
            id_conta = self.contas_map[self.conta_selecionada.get()]
            cat_info = self.categorias_map[self.categoria_selecionada.get()]
            id_categoria = cat_info['id']

            # Validação simples
            if valor <= 0 or not data or not descricao:
                messagebox.showwarning("Atenção", "Preencha todos os campos corretamente.")
                return

            if cat_info['tipo'] != tipo:
                 # Esta é uma validação de negócio que você pode adicionar: garante que uma Receita não use uma categoria de Despesa.
                 messagebox.showwarning("Atenção", f"A categoria '{self.categoria_selecionada.get()}' é do tipo '{cat_info['tipo']}', que não corresponde ao tipo selecionado '{tipo}'.")
                 return

            if self.db.inserir_transacao(id_conta, id_categoria, tipo, valor, data, descricao):
                messagebox.showinfo("Sucesso", "Transação registrada com sucesso!")
                # Limpa os campos após o sucesso
                self.valor_entry.set(0.0)
                self.descricao_entry.set("")

        except ValueError:
            messagebox.showerror("Erro de Valor", "Valor e Data devem estar no formato correto.")
        except KeyError:
            messagebox.showerror("Erro de Seleção", "Conta ou Categoria não selecionada/encontrada.")


    # --- ABA 2: GERENCIAMENTO DE CATEGORIAS ---

    def _criar_aba_categorias(self):
        self.frame_categorias = ttk.Frame(self.notebook, padding="15")
        self.notebook.add(self.frame_categorias, text='📁 Gerenciamento de Categorias')

        # Treeview para listar categorias
        ttk.Label(self.frame_categorias, text="Categorias Existentes", font=('Arial', 12, 'bold')).pack(pady=10)
        self.tree_categorias = ttk.Treeview(self.frame_categorias, columns=('ID', 'Nome', 'Tipo'), show='headings')
        self.tree_categorias.heading('ID', text='ID', anchor=tk.CENTER)
        self.tree_categorias.heading('Nome', text='Nome')
        self.tree_categorias.heading('Tipo', text='Tipo')
        self.tree_categorias.column('ID', width=50, anchor=tk.CENTER)
        self.tree_categorias.column('Nome', width=200)
        self.tree_categorias.column('Tipo', width=100)
        self.tree_categorias.pack(fill='both', expand=True)

        self._carregar_treeview_categorias()

        # Formulário de Adição
        frame_form = ttk.LabelFrame(self.frame_categorias, text="Adicionar Nova Categoria", padding=10)
        frame_form.pack(fill='x', pady=20)

        self.nova_cat_nome = tk.StringVar()
        self.nova_cat_tipo = tk.StringVar(value='DESPESA') # Default

        ttk.Label(frame_form, text="Nome:").grid(row=0, column=0, padx=5, pady=5, sticky='w')
        ttk.Entry(frame_form, textvariable=self.nova_cat_nome, width=30).grid(row=0, column=1, padx=5, pady=5, sticky='ew')

        ttk.Label(frame_form, text="Tipo:").grid(row=1, column=0, padx=5, pady=5, sticky='w')
        ttk.Radiobutton(frame_form, text='Receita', variable=self.nova_cat_tipo, value='RECEITA').grid(row=1, column=1, padx=5, pady=5, sticky='w')
        ttk.Radiobutton(frame_form, text='Despesa', variable=self.nova_cat_tipo, value='DESPESA').grid(row=1, column=2, padx=5, pady=5, sticky='w')

        ttk.Button(frame_form, text="Salvar Categoria", command=self._salvar_nova_categoria).grid(row=2, column=0, columnspan=3, pady=10)

        frame_form.columnconfigure(1, weight=1)

    def _carregar_treeview_categorias(self):
        """Carrega os dados do DB para o Treeview de Categorias."""
        # Limpar o Treeview
        for item in self.tree_categorias.get_children():
            self.tree_categorias.delete(item)

        categorias = self.db.buscar_categorias()
        for cat in categorias:
            # cat é uma tupla: (id, nome, tipo)
            self.tree_categorias.insert('', 'end', values=cat)

        self._carregar_dados_base() # Atualiza os mapas para o Combobox de Registro!

    def _salvar_nova_categoria(self):
        """Lógica para adicionar a nova categoria."""
        nome = self.nova_cat_nome.get().strip()
        tipo = self.nova_cat_tipo.get()

        if not nome:
            messagebox.showwarning("Atenção", "O nome da categoria não pode ser vazio.")
            return

        if self.db.adicionar_categoria(nome, tipo):
            messagebox.showinfo("Sucesso", f"Categoria '{nome}' adicionada.")
            self.nova_cat_nome.set("") # Limpa o campo
            self._carregar_treeview_categorias() # Atualiza a lista


    # --- ABA 3: CONSULTAS E RELATÓRIOS ---

    def _criar_aba_relatorios(self):
        frame_relatorios = ttk.Frame(self.notebook, padding="15")
        self.notebook.add(frame_relatorios, text='📊 Consultas e Relatórios')

        # --- Seção 1: Cálculo de Saldo Total ---

        frame_saldo = ttk.LabelFrame(frame_relatorios, text="Saldo Total", padding=10)
        frame_saldo.pack(fill='x', pady=10)

        self.saldo_label_var = tk.StringVar(value="Clique em 'Calcular Saldo' para ver o resultado.")

        ttk.Label(frame_saldo, text="Saldo Atual:", font=('Arial', 11, 'bold')).grid(row=0, column=0, padx=5, pady=5, sticky='w')
        ttk.Label(frame_saldo, textvariable=self.saldo_label_var, font=('Arial', 11)).grid(row=0, column=1, padx=5, pady=5, sticky='w')

        ttk.Button(frame_saldo, text="Calcular Saldo", command=self._calcular_e_exibir_saldo).grid(row=1, column=0, columnspan=2, pady=10)

        # --- Seção 2: Relatório por Categoria ---

        frame_relatorio_cat = ttk.LabelFrame(frame_relatorios, text="Transações por Categoria", padding=10)
        frame_relatorio_cat.pack(fill='both', expand=True, pady=10)

        # Seleção de Categoria
        categorias_nomes = list(self.categorias_map.keys())
        self.categoria_relatorio = tk.StringVar()

        ttk.Label(frame_relatorio_cat, text="Categoria:").grid(row=0, column=0, padx=5, pady=5, sticky='w')
        self.combo_rel_cat = ttk.Combobox(frame_relatorio_cat, textvariable=self.categoria_relatorio, values=categorias_nomes, state='readonly')
        self.combo_rel_cat.grid(row=0, column=1, padx=5, pady=5, sticky='ew')

        ttk.Button(frame_relatorio_cat, text="Ver Transações", command=self._carregar_transacoes_por_categoria).grid(row=0, column=2, padx=10, pady=5)

        frame_relatorio_cat.columnconfigure(1, weight=1)

        # Treeview para o Relatório
        ttk.Label(frame_relatorio_cat, text="Resultados:", font=('Arial', 10)).grid(row=1, column=0, columnspan=3, pady=10, sticky='w')
        self.tree_relatorio = ttk.Treeview(frame_relatorio_cat, columns=('Data', 'Categoria', 'Valor', 'Tipo', 'Conta'), show='headings')
        self.tree_relatorio.heading('Data', text='Data')
        self.tree_relatorio.heading('Categoria', text='Categoria')
        self.tree_relatorio.heading('Valor', text='Valor (R$)')
        self.tree_relatorio.heading('Tipo', text='Tipo')
        self.tree_relatorio.heading('Conta', text='Conta')

        self.tree_relatorio.column('Data', width=80, anchor=tk.CENTER)
        self.tree_relatorio.column('Valor', width=80, anchor=tk.E)
        self.tree_relatorio.grid(row=2, column=0, columnspan=3, sticky='nsew', padx=5, pady=5)
        frame_relatorio_cat.rowconfigure(2, weight=1)
        frame_relatorio_cat.columnconfigure(0, weight=1)

    def _calcular_e_exibir_saldo(self):
        """Chama a agregação SQL e atualiza o Label."""
        saldo = self.db.calcular_saldo_total()
        cor = 'green' if saldo >= 0 else 'red'
        self.saldo_label_var.set(f"R$ {saldo:,.2f}".replace('.', '#').replace(',', '.').replace('#', ','))

        # O Tkinter não tem uma maneira direta de mudar a cor de um widget ttk.Label
        # Se usarmos um tk.Label simples:
        # self.master.nametowidget(self.saldo_label_var.get()).config(fg=cor)
        # Vamos apenas deixar o texto, mas em um ambiente real usaríamos bibliotecas como ttkthemes ou tk.Label para isso.

        messagebox.showinfo("Resultado", f"O Saldo Total é: R$ {saldo:,.2f}".replace('.', '#').replace(',', '.').replace('#', ','))


    def _carregar_transacoes_por_categoria(self):
        """Busca e exibe transações da categoria selecionada."""
        # Limpar o Treeview
        for item in self.tree_relatorio.get_children():
            self.tree_relatorio.delete(item)

        nome_categoria = self.categoria_relatorio.get()
        if not nome_categoria:
            return

        categoria_id = self.categorias_map[nome_categoria]['id']

        transacoes = self.db.buscar_transacoes_por_categoria(categoria_id)

        for t in transacoes:
            # t: (data, nome_categoria, valor, tipo, nome_conta)
            self.tree_relatorio.insert('', 'end', values=t)

        if not transacoes:
            messagebox.showinfo("Consulta", f"Nenhuma transação encontrada para a categoria '{nome_categoria}'.")

# --- Bloco de Execução Principal ---
if __name__ == "__main__":
    try:
        root = tk.Tk()
        app = AppGUI(root)
        root.mainloop()
    except Exception as e:
        # Garante que qualquer erro no Tkinter ou DB seja exibido
        print(f"Ocorreu um erro fatal: {e}")
        messagebox.showerror("Erro Fatal", f"O aplicativo encontrou um erro: {e}")

Ocorreu um erro fatal: no display name and no $DISPLAY environment variable


TclError: no display name and no $DISPLAY environment variable

**PARTE 2 DO PROJETO**